In [ ]:
# 1. Install dependencies
%pip -q install pandas scikit-learn joblib requests
print("Dependencies installed.")

Dependencies installed.


In [ ]:
# 2. Get the dataset
# Option A (recommended): upload kidney_disease.csv from your course materials.
from google.colab import files
import os, pandas as pd

if not os.path.exists("kidney_disease.csv"):
    print("Upload kidney_disease.csv now.")
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if csv_files:
        chosen = "kidney_disease.csv" if "kidney_disease.csv" in csv_files else csv_files[0]
        if chosen != "kidney_disease.csv":
            os.rename(chosen, "kidney_disease.csv")

df = pd.read_csv("kidney_disease.csv")
print("Dataset loaded. Shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())

Upload kidney_disease.csv now.


Saving 1.%20kidney_disease.csv to 1.%20kidney_disease.csv
Dataset loaded. Shape: (400, 26)


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd



Columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


In [ ]:
# 3. Validate expected columns and inspect data
df.columns = [str(c).strip() for c in df.columns]
if "classification" not in df.columns:
    raise ValueError(
        "Expected target column 'classification' was not found. "
        f"Available columns: {df.columns.tolist()}. Use the course dataset or update TARGET below."
    )

df["classification"] = df["classification"].astype(str).str.strip()
print("Target distribution:")
display(df["classification"].value_counts(dropna=False).to_frame("count"))
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count"))
print("\nDataset shape:", df.shape)

Target distribution:


,count
classification,
ckd,250
notckd,150



Missing values per column:


,missing_count
id,0
age,9
bp,12
sg,47
al,46
su,49
rbc,152
pc,65
pcc,4
ba,4



Dataset shape: (400, 26)


In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

TARGET = "classification"
X = df.drop(columns=[TARGET]).copy()
if "id" in X.columns:
    X = X.drop(columns=["id"])
y = df[TARGET].copy()

# Make mixed/messy CSV columns more robust while preserving categorical values.
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype(object).apply(lambda v: v.strip() if isinstance(v, str) else v).replace({"?": float("nan"), "": float("nan")})

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Stratification is useful when each target class has enough samples.
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
except ValueError:
    print("Warning: stratified split unavailable; using regular split.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
model = RandomForestClassifier(n_estimators=100, random_state=42)
ml_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Training model...")
ml_pipeline.fit(X_train, y_train)
y_pred = ml_pipeline.predict(X_test)

print("\nMODEL EVALUATION")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

MODEL_FILE = "kidney_disease_model.pkl"
joblib.dump(ml_pipeline, MODEL_FILE)
print(f"\nSaved complete pipeline: {MODEL_FILE} ({os.path.getsize(MODEL_FILE)/1024:.2f} KB)")

Numeric features: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo']
Categorical features: ['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
Training model...

MODEL EVALUATION
Accuracy : 0.9875
Precision: 0.9877
Recall   : 0.9875
F1-score : 0.9875

Classification report:
              precision    recall  f1-score   support

         ckd       0.98      1.00      0.99        50
      notckd       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80


Saved complete pipeline: kidney_disease_model.pkl (337.81 KB)


In [ ]:
# Reload the saved artifact, as a deployed script would do
loaded_model = joblib.load("kidney_disease_model.pkl")

example_values = {
    "age": 48, "bp": 80, "sg": 1.02, "al": 1, "su": 0,
    "rbc": "normal", "pc": "normal", "pcc": "notpresent", "ba": "notpresent",
    "bgr": 121, "bu": 36, "sc": 1.2, "sod": 138, "pot": 4.5, "hemo": 15.4,
    "pcv": "44", "wc": "7800", "rc": "5.2", "htn": "no", "dm": "no",
    "cad": "no", "appet": "good", "pe": "no", "ane": "no"
}

# Start from a real row to guarantee every expected feature is present.
online_record = X.iloc[[0]].copy()
for col, value in example_values.items():
    if col in online_record.columns:
        online_record.loc[online_record.index[0], col] = value

online_prediction = loaded_model.predict(online_record)
print("Input record:")
display(online_record)
print("Online prediction:", online_prediction[0])
print("Note: this is a classroom model output, not medical advice.")

Input record:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane
0,48.0,80.0,1.02,1.0,0.0,normal,normal,notpresent,notpresent,121.0,...,15.4,44,7800,5.2,no,no,no,good,no,no


Online prediction: notckd
Note: this is a classroom model output, not medical advice.


In [ ]:
# Use the same feature schema as training
batch_X = df.drop(columns=[TARGET]).copy()
ids = batch_X["id"].copy() if "id" in batch_X.columns else pd.Series(range(1, len(batch_X) + 1), name="id")
if "id" in batch_X.columns:
    batch_X = batch_X.drop(columns=["id"])
for col in batch_X.columns:
    if col in X.columns and X[col].dtype == "object":
        batch_X[col] = batch_X[col].astype(object).apply(lambda v: v.strip() if isinstance(v, str) else v).replace({"?": float("nan"), "": float("nan")})

batch_predictions = loaded_model.predict(batch_X)
batch_output = pd.DataFrame({"id": ids.values, "prediction": batch_predictions})
batch_output.to_csv("batch_predictions.csv", index=False)

print("Batch prediction completed.")
print("Number of records:", len(batch_output))
display(batch_output.head(10))
print("Output file created: batch_predictions.csv")

Batch prediction completed.
Number of records: 400


,id,prediction
0,0,ckd
1,1,ckd
2,2,ckd
3,3,ckd
4,4,ckd
5,5,ckd
6,6,ckd
7,7,ckd
8,8,ckd
9,9,ckd


Output file created: batch_predictions.csv


In [ ]:
# Optional: download the batch prediction output to your computer
from google.colab import files
files.download("batch_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time

stream_X = df.drop(columns=[TARGET, "id"], errors="ignore").copy()
for col in stream_X.columns:
    if col in X.columns and X[col].dtype == "object":
        stream_X[col] = stream_X[col].astype(object).apply(lambda v: v.strip() if isinstance(v, str) else v).replace({"?": float("nan"), "": float("nan")})

N_RECORDS = min(10, len(stream_X))  # Exercise: change to 20 if dataset has enough rows
DELAY_SECONDS = 0.5                  # Exercise: change to 1.0 and compare
print("Starting streaming prediction...\n")
for i in range(N_RECORDS):
    record = stream_X.iloc[[i]]
    prediction = loaded_model.predict(record)
    print(f"Incoming Record {i + 1} -> Prediction: {prediction[0]}")
    time.sleep(DELAY_SECONDS)
print("\nStreaming simulation completed.")

Starting streaming prediction...

Incoming Record 1 -> Prediction: ckd
Incoming Record 2 -> Prediction: ckd
Incoming Record 3 -> Prediction: ckd
Incoming Record 4 -> Prediction: ckd
Incoming Record 5 -> Prediction: ckd
Incoming Record 6 -> Prediction: ckd
Incoming Record 7 -> Prediction: ckd
Incoming Record 8 -> Prediction: ckd
Incoming Record 9 -> Prediction: ckd
Incoming Record 10 -> Prediction: ckd

Streaming simulation completed.
